# Phase I · Mask low-expression model genes per cell

This notebook uses the model-gene prepared H5AD created by `download_sampled_cells3.ipynb`. It masks the lowest-expressed fraction of detected model genes in each cell, reruns CellTypist, and compares the result with an unmasked control from the exact same input.

The input `.X` should contain log1p-normalized expression; `layers["counts"]` should contain raw counts. Masking is applied to `.X` after normalization and does not renormalize the remaining genes. The separate control is important because the older all-gene baseline used a different normalization basis.


## 1. Configure the input and masking run

Set `MASK_FRACTION` to the fraction of each cell's detected model genes to mask. For example, `0.50` masks the lowest-expression half; `0` is an unmasked run. The source file should already contain only genes matched to CellTypist.


In [1]:
from pathlib import Path

import anndata as ad
import celltypist
from celltypist import models
import numpy as np
import pandas as pd
from scipy.sparse import issparse, csr_matrix

DATA_ROOT = Path(
    "/Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE"
)
DATA_DIR = DATA_ROOT / "iha_celltypist"
INPUT_H5AD = DATA_DIR / "iha_expression_model_genes_prepared.h5ad"
MODEL_NAME = "Immune_All_Low.pkl"

MASK_FRACTION = 0.50
SEED = 0
BATCH_SIZE = 1_000
CELL_KEY = "soma_joinid"

if not 0 <= MASK_FRACTION <= 1:
    raise ValueError("MASK_FRACTION must be between 0 and 1.")
if BATCH_SIZE < 1:
    raise ValueError("BATCH_SIZE must be positive.")
if not INPUT_H5AD.exists():
    raise FileNotFoundError(f"Prepared model-gene H5AD not found: {INPUT_H5AD}")

OUT_DIR = DATA_DIR / "model_gene_masking"
RUN_DIR = OUT_DIR / f"mask_{MASK_FRACTION:g}_seed_{SEED}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input: {INPUT_H5AD}")
print(f"Run output: {RUN_DIR}")


/Users/artsvetkova/miniconda3/envs/sc_immune/lib/python3.10/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


Input: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/iha_expression_model_genes_prepared.h5ad
Run output: /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/model_gene_masking/mask_0.5_seed_0


## 2. Load and validate the prepared model-gene dataset

This file already contains the model genes available in the atlas, so this notebook does not filter genes or create another subset file. It checks the model-feature overlap, cell IDs, normalized expression, and retained raw-count layer before masking.


In [2]:
# Download the model once if it is not already cached, then load it.
# models.download_models(model=MODEL_NAME)
model = models.Model.load(MODEL_NAME)
model_genes = pd.Index(pd.Series(model.features).astype(str).unique())

adata = ad.read_h5ad(INPUT_H5AD)
if CELL_KEY not in adata.obs.columns:
    raise KeyError(f"Input is missing obs[{CELL_KEY!r}].")
if "counts" not in adata.layers:
    raise KeyError("Input is missing layers['counts'] with raw counts.")

symbols = pd.Index(adata.var_names.astype(str))
if not symbols.is_unique:
    raise ValueError("Input var_names must be unique gene symbols.")
not_in_model = symbols.difference(model_genes)
if len(not_in_model):
    raise ValueError(
        f"Input contains {len(not_in_model)} genes that are not model features. "
        f"Examples: {not_in_model[:10].tolist()}"
    )

numeric_cell_ids = pd.to_numeric(adata.obs[CELL_KEY], errors="raise")
if numeric_cell_ids.isna().any():
    raise ValueError("Input contains missing cell IDs.")
cell_ids = numeric_cell_ids.to_numpy(dtype=np.int64)
if len(np.unique(cell_ids)) != len(cell_ids):
    raise ValueError("Input contains duplicate soma_joinid values.")
adata.obs_names = pd.Index(cell_ids.astype(str), name="cell_id")

x_values = adata.X.data if issparse(adata.X) else np.asarray(adata.X).ravel()
if not np.isfinite(x_values).all() or (x_values < 0).any():
    raise ValueError("Expected finite, nonnegative log1p expression in .X.")

# Spot-check the normalized scale: expm1(log1p(counts)) should sum to ~10,000.
first_row = adata.X[0]
first_values = first_row.data if issparse(first_row) else np.asarray(first_row).ravel()
first_row_total = np.expm1(first_values).sum()
if not np.isclose(first_row_total, 10_000, atol=1):
    raise ValueError(
        f"Input does not appear normalized to 10,000 counts per cell "
        f"(first-cell total: {first_row_total:.2f})."
    )

missing_model_genes = model_genes.difference(symbols)
print(adata)
print(f"Model features: {len(model_genes):,}")
print(f"Features present in input: {len(symbols):,}")
print(f"Model features absent from this atlas: {len(missing_model_genes):,}")
print(f"First-cell pre-log normalized total: {first_row_total:,.1f}")


AnnData object with n_obs × n_vars = 54225 × 6089
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'age', 'age_group'
    var: 'soma_joinid', 'feature_id', 'feature_name', 'feature_type', 'feature_length', 'nnz', 'n_measured_obs'
    uns: 'log1p'
    layers: 'counts'
Model features: 6,639
Features present in input: 6,089
Model features absent from this atlas: 550
First-cell pre-log normalized total: 10,000.0


## 3. Mask the lowest-expression detected genes per cell

For each cell, the code masks `floor(MASK_FRACTION × number of detected genes)` genes with the lowest expression. Ties are broken reproducibly using `SEED`. Genes already at zero remain zero. The remaining expression values are not renormalized, so this isolates the effect of removing those inputs. CellTypist may warn that a perturbed matrix no longer sums to 10,000 after reversing log1p; that is expected for the masked run.


In [3]:
def mask_low_expression(matrix, fraction, seed):
    X = csr_matrix(matrix, copy=True)
    X.sum_duplicates()
    X.sort_indices()
    X.eliminate_zeros()

    if not np.isfinite(X.data).all() or (X.data < 0).any():
        raise ValueError("Expected finite, nonnegative log1p expression in .X.")

    n_expressed = np.diff(X.indptr)
    n_masked = np.floor(fraction * n_expressed).astype(int)
    rng = np.random.default_rng(seed)

    for row, count in enumerate(n_masked):
        if count == 0:
            continue
        values = X.data[X.indptr[row] : X.indptr[row + 1]]
        shuffled = rng.permutation(values.size)
        lowest_first = np.argsort(values[shuffled], kind="stable")
        values[shuffled[lowest_first[:count]]] = 0

    X.eliminate_zeros()
    return X, n_expressed, n_masked


masked_X, n_expressed, n_masked = mask_low_expression(
    adata.X, MASK_FRACTION, SEED
)
masked = adata.copy()
masked.X = masked_X
masked.obs["n_expressed_model_genes"] = n_expressed
masked.obs["n_genes_masked"] = n_masked
masked.uns["masking"] = {
    "fraction": float(MASK_FRACTION),
    "seed": int(SEED),
    "model": MODEL_NAME,
    "selection": "lowest expressed detected model genes per cell",
    "renormalized_after_masking": False,
}

masked_path = RUN_DIR / "masked_expression.h5ad"
masked.write_h5ad(masked_path, compression="gzip")
print(f"Masked {n_masked.sum():,} of {n_expressed.sum():,} detected cell-gene entries.")
print(f"Median genes masked per cell: {np.median(n_masked):,.0f}")
print(f"Saved {masked_path}")


Masked 22,644,858 of 45,316,965 detected cell-gene entries.
Median genes masked per cell: 389
Saved /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/model_gene_masking/mask_0.5_seed_0/masked_expression.h5ad


## 4. Run an unmasked control and the masked input through CellTypist

The unmasked control is classified from the same prepared file and with the same model and batch size. Compare it with the masked result to isolate changes caused by this mask. Do not use the older all-gene baseline as the direct control because it was normalized over a different gene set.


In [4]:
print(f"Classifying the unmasked control: {adata.n_obs:,} cells")
control_result = celltypist.annotate(
    adata,
    model=model,
    majority_voting=False,
)

print(f"Classifying the masked input: {masked.n_obs:,} cells")
masked_result = celltypist.annotate(
    masked,
    model=model,
    majority_voting=False,
)

control_labels = control_result.predicted_labels.copy()
control_scores = control_result.decision_matrix.copy()
masked_labels = masked_result.predicted_labels.copy()
masked_scores = masked_result.decision_matrix.copy()

expected_ids = pd.Index(adata.obs_names.astype(str))
def align_celltypist_output(table, name):
    table = table.copy()
    table.index = table.index.astype(str)
    if table.index.has_duplicates:
        raise AssertionError(f"{name} contains duplicate cell IDs.")
    missing = expected_ids.difference(table.index)
    extra = table.index.difference(expected_ids)
    if len(missing) or len(extra):
        raise AssertionError(
            f"{name} cell IDs do not match the input. "
            f"Missing: {len(missing):,}; extra: {len(extra):,}."
        )
    table = table.reindex(expected_ids)
    table.index.name = CELL_KEY
    return table


control_labels = align_celltypist_output(control_labels, "unmasked predictions")
control_scores = align_celltypist_output(control_scores, "unmasked decision matrix")
masked_labels = align_celltypist_output(masked_labels, "masked predictions")
masked_scores = align_celltypist_output(masked_scores, "masked decision matrix")

control_labels.to_csv(RUN_DIR / "unmasked_control_predictions.csv")
control_scores.to_csv(RUN_DIR / "unmasked_control_decision_matrix.csv")

mask_diagnostics = masked.obs[["n_expressed_model_genes", "n_genes_masked"]]
masked_export = masked_labels.join(mask_diagnostics)
masked_export.index.name = CELL_KEY
masked_export.to_csv(RUN_DIR / "masked_predictions.csv")
masked_scores.to_csv(RUN_DIR / "masked_decision_matrix.csv")

label_column = "predicted_labels" if "predicted_labels" in control_labels.columns else control_labels.columns[0]
comparison = adata.obs[
    [column for column in [CELL_KEY, "donor_id", "age_group", "cell_type"] if column in adata.obs.columns]
].copy()
comparison["unmasked_label"] = control_labels[label_column].astype(str).to_numpy()
comparison["masked_label"] = masked_labels[label_column].astype(str).to_numpy()
comparison["label_changed"] = comparison["unmasked_label"] != comparison["masked_label"]
comparison.index.name = "cell_id"
comparison.to_csv(RUN_DIR / "per_cell_prediction_comparison.csv", index=False)

n_changed = int(comparison["label_changed"].sum())
print(f"Label changed for {n_changed:,} of {len(comparison):,} cells ({n_changed / len(comparison):.1%}).")
print(f"Saved masked and unmasked predictions, decision matrices, and comparison to {RUN_DIR.resolve()}")
comparison.head()


🔬 Input data has 54225 cells and 6089 genes
🔗 Matching reference genes in the model
🧬 6089 features used for prediction


Classifying the unmasked control: 54,225 cells


⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
⚠️ Warning: invalid expression matrix, expect ALL genes and log1p normalized expression to 10000 counts per cell. The prediction result may not be accurate
🔬 Input data has 54225 cells and 6089 genes
🔗 Matching reference genes in the model
🧬 6089 features used for prediction


Classifying the masked input: 54,225 cells


⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!


Label changed for 10,140 of 54,225 cells (18.7%).
Saved masked and unmasked predictions, decision matrices, and comparison to /Users/artsvetkova/Studies/_JKU/3rd_semester/AI and Visualization/UE/iha_celltypist/model_gene_masking/mask_0.5_seed_0


,soma_joinid,donor_id,age_group,cell_type,unmasked_label,masked_label,label_changed
cell_id,,,,,,,
103847865,103847865,BR1005,18-45,"central memory CD4-positive, alpha-beta T cell",Tcm/Naive helper T cells,Tcm/Naive helper T cells,False
103847873,103847873,BR1005,18-45,T cell,Tem/Temra cytotoxic T cells,Tem/Temra cytotoxic T cells,False
103847892,103847892,BR1005,18-45,"naive thymus-derived CD8-positive, alpha-beta ...",Tcm/Naive cytotoxic T cells,Tcm/Naive cytotoxic T cells,False
103847893,103847893,BR1005,18-45,"naive thymus-derived CD8-positive, alpha-beta ...",Tcm/Naive cytotoxic T cells,Tcm/Naive cytotoxic T cells,False
103847900,103847900,BR1005,18-45,intermediate monocyte,Classical monocytes,Classical monocytes,False
